# 🔬 Cancer Clinical Trial Outcome Predictor — Complete Training Notebook

This notebook is the **single source of truth** for the model used in the Flask web application.

| Item | Detail |
|------|--------|
| **Flask dataset** | Synthetic — 3,000 rows generated by `generate_model.py` |
| **Real dataset (Part B)** | ClinicalTrials.gov REST API v2 (public, no key needed) |
| **Algorithm** | Gradient Boosting Classifier inside a sklearn `Pipeline` |
| **Saved artifacts** | `model_pipeline.pkl`, `feature_columns.json` |
| **scaler.pkl** | ❌ Not saved separately — the `StandardScaler` lives *inside* `model_pipeline.pkl` |

---

## ⚠️ Honest explanation of the Flask project dataset

The Flask deployment demo was built to demonstrate **pipeline architecture** (form → preprocessing → model → result).  
It used **synthetically generated data** — not real hospital records.  
The synthetic labels were produced by explicit domain-knowledge rules (e.g. Phase III + industry sponsor → higher success probability).  
**This notebook reproduces that training exactly** so you can verify every step, then shows how to upgrade to real ClinicalTrials.gov data.

---
**Run all cells top-to-bottom. Runtime ≈ 3 min (synthetic) or 8 min (real data).**


## 📦 Cell 1 — Install & import libraries


In [ ]:
# All packages are pre-installed in Colab except imbalanced-learn
# Uncomment the line below only if you need it
# !pip install imbalanced-learn --quiet

import warnings, json, os, time, requests
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection   import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder
from sklearn.ensemble          import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.metrics           import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

print("✅  Libraries loaded.")
print(f"   numpy  {np.__version__}")
print(f"   pandas {pd.__version__}")


---
# PART A — Reproduce the EXACT Flask project model (synthetic data)

This reproduces `generate_model.py` character-for-character.  
Running this section will produce **identical `model_pipeline.pkl` and `feature_columns.json`**  
to the files already inside the Flask app's `model/` folder.


## 🗂️ Cell 2 — Dataset: what it is and how it was generated


In [ ]:
# ══════════════════════════════════════════════════════════════════
# DATASET: SYNTHETIC (3,000 rows)
# Source : generate_model.py  (shipped inside the Flask project zip)
# No external file download needed — data is generated here in code.
#
# Label engineering:
#   success_prob is calculated from domain-knowledge rules:
#     • Phase III/IV  →  higher probability
#     • targeted_therapy / immunotherapy  →  higher probability
#     • pancreatic cancer  →  lower probability
#     • industry sponsor  →  slightly higher
#   Gaussian noise (σ=0.10) is added, then thresholded at 0.50
#   outcome = 1 (success) if success_prob >= 0.50, else 0 (failure)
# ══════════════════════════════════════════════════════════════════

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Feature value pools ────────────────────────────────────────────
CANCER_TYPES  = ["lung","breast","colorectal","prostate","leukemia",
                 "melanoma","pancreatic","ovarian","bladder","renal"]
DRUG_TYPES    = ["chemotherapy","immunotherapy","targeted_therapy",
                 "hormone_therapy","radiation","combination"]
PHASES        = ["I","II","III","IV"]
SPONSOR_TYPES = ["industry","academic","government"]

CATEGORICAL_COLS = ["cancer_type","drug_type","phase","sponsor_type"]
NUMERICAL_COLS   = ["enrollment_size","trial_duration_months"]
ALL_FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS  # ORDER MUST NEVER CHANGE

# ── Generate 3,000 rows ────────────────────────────────────────────
N = 3000

cancer_type           = np.random.choice(CANCER_TYPES, N)
drug_type             = np.random.choice(DRUG_TYPES, N)
phase                 = np.random.choice(PHASES, N, p=[0.15, 0.35, 0.40, 0.10])
enrollment_size       = np.random.randint(20, 2000, N)
trial_duration_months = np.random.randint(6, 120, N)
sponsor_type          = np.random.choice(SPONSOR_TYPES, N, p=[0.55, 0.30, 0.15])

# ── Rule-based label engineering ──────────────────────────────────
success_prob = np.full(N, 0.40)

success_prob += np.where(phase == "I",   -0.10,
               np.where(phase == "II",    0.00,
               np.where(phase == "III",   0.10, 0.05)))

success_prob += np.where(drug_type == "targeted_therapy",  0.12,
               np.where(drug_type == "immunotherapy",       0.08,
               np.where(drug_type == "combination",         0.06,
               np.where(drug_type == "hormone_therapy",     0.04,
               np.where(drug_type == "chemotherapy",       -0.02, -0.05)))))

success_prob += np.where(cancer_type == "breast",    0.10,
               np.where(cancer_type == "prostate",   0.08,
               np.where(cancer_type == "leukemia",   0.05,
               np.where(cancer_type == "pancreatic",-0.12,
               np.where(cancer_type == "lung",      -0.05, 0.00)))))

success_prob += np.clip((enrollment_size - 200)          / 5000, -0.05, 0.05)
success_prob += np.clip((trial_duration_months - 24)     / 500,  -0.03, 0.03)
success_prob += np.where(sponsor_type == "industry",  0.05,
               np.where(sponsor_type == "academic",   0.00, -0.03))

# Add Gaussian noise and threshold
success_prob = np.clip(success_prob + np.random.normal(0, 0.10, N), 0.05, 0.95)
outcome      = (success_prob >= 0.50).astype(int)

# ── Build DataFrame ────────────────────────────────────────────────
df_synth = pd.DataFrame({
    "cancer_type"           : cancer_type,
    "drug_type"             : drug_type,
    "phase"                 : phase,
    "enrollment_size"       : enrollment_size.astype(float),
    "trial_duration_months" : trial_duration_months.astype(float),
    "sponsor_type"          : sponsor_type,
    "outcome"               : outcome,
})

print(f"✅  Synthetic dataset generated: {df_synth.shape}")
print(f"\nOutcome distribution:")
print(df_synth["outcome"].value_counts())
print(f"\nPositive rate: {df_synth['outcome'].mean():.2%}")
print(f"\nFirst 5 rows:")
df_synth.head()


## 📊 Cell 3 — Exploratory Data Analysis


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Synthetic Dataset — Feature Distributions", fontsize=14, fontweight="bold")

# 1. Outcome balance
axes[0,0].bar(["Failure (0)", "Success (1)"],
              df_synth["outcome"].value_counts().sort_index(),
              color=["#ef4444","#22c55e"])
axes[0,0].set_title("Outcome balance")
axes[0,0].set_ylabel("Count")

# 2. Phase distribution
phase_counts = df_synth["phase"].value_counts().reindex(["I","II","III","IV"])
axes[0,1].bar(phase_counts.index, phase_counts.values, color="#38bdf8")
axes[0,1].set_title("Trial phase distribution")

# 3. Success rate by phase
sr_phase = df_synth.groupby("phase")["outcome"].mean().reindex(["I","II","III","IV"])
axes[0,2].bar(sr_phase.index, sr_phase.values, color="#6366f1")
axes[0,2].set_title("Success rate by phase")
axes[0,2].set_ylabel("Success rate")
axes[0,2].set_ylim(0, 1)
for i, v in enumerate(sr_phase.values):
    axes[0,2].text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)

# 4. Success rate by drug type
sr_drug = df_synth.groupby("drug_type")["outcome"].mean().sort_values()
axes[1,0].barh(sr_drug.index, sr_drug.values, color="#f59e0b")
axes[1,0].set_title("Success rate by drug type")
axes[1,0].set_xlabel("Success rate")

# 5. Enrollment distribution
axes[1,1].hist(df_synth["enrollment_size"], bins=40, color="#14b8a6", edgecolor="white")
axes[1,1].set_title("Enrollment size distribution")
axes[1,1].set_xlabel("Participants")

# 6. Success rate by cancer type
sr_cancer = df_synth.groupby("cancer_type")["outcome"].mean().sort_values()
axes[1,2].barh(sr_cancer.index, sr_cancer.values, color="#a78bfa")
axes[1,2].set_title("Success rate by cancer type")
axes[1,2].set_xlabel("Success rate")

plt.tight_layout()
plt.savefig("/content/eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/eda_plots.png")


## 🧹 Cell 4 — Data preprocessing


In [ ]:
# The synthetic data is already clean.
# For reproducibility, we apply the same steps that would be
# applied to real data: type casting, value validation.

df = df_synth.copy()

# Ensure correct dtypes
df["enrollment_size"]       = df["enrollment_size"].astype(float)
df["trial_duration_months"] = df["trial_duration_months"].astype(float)
df["outcome"]               = df["outcome"].astype(int)

# Validate categorical values are in expected sets
assert set(df["phase"].unique()).issubset(set(PHASES)), "Unexpected phase value"
assert set(df["sponsor_type"].unique()).issubset(set(SPONSOR_TYPES)), "Unexpected sponsor"

print("✅  Preprocessing complete.")
print(f"   Rows: {len(df)}")
print(f"   Dtypes:\n{df.dtypes}")
print(f"\nBasic statistics:")
print(df[NUMERICAL_COLS].describe().round(2))


## ⚙️ Cell 5 — Feature engineering


In [ ]:
# The Flask project uses EXACTLY these 6 features — no more.
# ALL encoding and scaling is handled INSIDE the sklearn Pipeline,
# so no manual feature engineering happens before fitting.
#
# Features passed to the Pipeline:
#   Categorical  →  OneHotEncoder  (inside Pipeline)
#     • cancer_type    10 unique values
#     • drug_type       6 unique values
#     • phase           4 unique values  (I / II / III / IV)
#     • sponsor_type    3 unique values
#
#   Numerical  →  StandardScaler  (inside Pipeline)
#     • enrollment_size          (range 20–2000)
#     • trial_duration_months    (range 6–120)
#
# NOTE on scaler.pkl:
#   There is NO separate scaler.pkl in this project.
#   The StandardScaler is embedded inside model_pipeline.pkl.
#   To extract it if needed:
#       import joblib
#       pipe = joblib.load("model_pipeline.pkl")
#       scaler = pipe.named_steps["preprocessor"].named_transformers_["num"]

X = df[ALL_FEATURE_COLS]
y = df["outcome"]

print("Feature matrix shape:", X.shape)
print("Label distribution:")
print(y.value_counts())
print()
print("Categorical columns:", CATEGORICAL_COLS)
print("Numerical columns  :", NUMERICAL_COLS)
print("Feature order (must match Flask app exactly):", ALL_FEATURE_COLS)


## ✂️ Cell 6 — Train / test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.20,      # 80% train, 20% test
    random_state = 42,        # reproducible split
    stratify     = y,         # preserve class ratio
)

print(f"Training set : {len(X_train):,} rows  ({len(X_train)/len(X):.0%})")
print(f"Test set     : {len(X_test):,} rows   ({len(X_test)/len(X):.0%})")
print(f"\nClass balance in train: {y_train.value_counts().to_dict()}")
print(f"Class balance in test : {y_test.value_counts().to_dict()}")


## 🤖 Cell 7 — Build sklearn Pipeline & train model


In [ ]:
# ══════════════════════════════════════════════════════════════════
# PIPELINE ARCHITECTURE
# ══════════════════════════════════════════════════════════════════
#
#  Input DataFrame (6 columns)
#       ↓
#  ColumnTransformer
#    ├── OneHotEncoder  →  cancer_type, drug_type, phase, sponsor_type
#    └── StandardScaler →  enrollment_size, trial_duration_months
#       ↓
#  GradientBoostingClassifier
#    • n_estimators  = 300
#    • learning_rate = 0.05
#    • max_depth     = 4
#    • subsample     = 0.8
#    • random_state  = 42
#       ↓
#  predict_proba() → P(success)
#
# WHY A SINGLE PIPELINE?
#   If preprocessing were done separately from the model,
#   you'd need to save a separate scaler.pkl, apply it manually
#   during prediction, and risk mismatching transformations.
#   One Pipeline object guarantees training == deployment always.
# ══════════════════════════════════════════════════════════════════

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ("num", StandardScaler(),                                              NUMERICAL_COLS),
    ],
    remainder="drop",
)

full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   GradientBoostingClassifier(
        n_estimators  = 300,
        learning_rate = 0.05,
        max_depth     = 4,
        subsample     = 0.8,
        random_state  = 42,
    )),
])

print("Training model …")
full_pipeline.fit(X_train, y_train)
print("✅  Training complete.")
print()
print("Pipeline steps:")
for name, step in full_pipeline.steps:
    print(f"  {name}: {type(step).__name__}")
print()
print("Preprocessor transformers:")
for name, trans, cols in full_pipeline.named_steps["preprocessor"].transformers_:
    print(f"  {name}: {type(trans).__name__} on {cols}")


## 📊 Cell 8 — Model evaluation metrics


In [ ]:
y_pred      = full_pipeline.predict(X_test)
y_pred_prob = full_pipeline.predict_proba(X_test)[:, 1]

acc     = accuracy_score (y_test, y_pred)
prec    = precision_score(y_test, y_pred,      zero_division=0)
rec     = recall_score   (y_test, y_pred,      zero_division=0)
f1      = f1_score       (y_test, y_pred,      zero_division=0)
roc_auc = roc_auc_score  (y_test, y_pred_prob)

print("╔══════════════════════════════════════════╗")
print("║     MODEL EVALUATION — TEST SET          ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Accuracy   : {acc:.4f}  ({acc*100:.2f}%)            ║")
print(f"║  Precision  : {prec:.4f}                        ║")
print(f"║  Recall     : {rec:.4f}                        ║")
print(f"║  F1 Score   : {f1:.4f}                        ║")
print(f"║  ROC-AUC    : {roc_auc:.4f}                        ║")
print("╚══════════════════════════════════════════╝")
print()
print("Per-class breakdown:")
print(classification_report(
    y_test, y_pred,
    target_names=["Failure (0)", "Success (1)"],
    zero_division=0
))

# 5-fold cross-validation
print("5-fold stratified cross-validation (ROC-AUC) …")
cv       = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs  = cross_val_score(full_pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"  Fold scores : {cv_aucs.round(4)}")
print(f"  Mean ± Std  : {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}")


## 🏆 Cell 9 — Compare against baseline models


In [ ]:
results = {}

# Logistic Regression baseline
lr_pipe = Pipeline([
    ("pre", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ("num", StandardScaler(), NUMERICAL_COLS),
    ], remainder="drop")),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])
lr_pipe.fit(X_train, y_train)
results["Logistic Regression"] = {
    "acc": accuracy_score(y_test, lr_pipe.predict(X_test)),
    "auc": roc_auc_score(y_test,  lr_pipe.predict_proba(X_test)[:,1]),
}

# Random Forest baseline
rf_pipe = Pipeline([
    ("pre", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ("num", StandardScaler(), NUMERICAL_COLS),
    ], remainder="drop")),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42)),
])
rf_pipe.fit(X_train, y_train)
results["Random Forest"] = {
    "acc": accuracy_score(y_test, rf_pipe.predict(X_test)),
    "auc": roc_auc_score(y_test,  rf_pipe.predict_proba(X_test)[:,1]),
}

# Our GBM
results["Gradient Boosting ✓"] = {"acc": acc, "auc": roc_auc}

print(f"{'Model':<28} {'Accuracy':>10} {'ROC-AUC':>10}")
print("-" * 50)
for name, m in results.items():
    mark = " ← best" if "✓" in name else ""
    print(f"{name:<28} {m['acc']:>10.4f} {m['auc']:>10.4f}{mark}")


## 📈 Cell 10 — Evaluation plots


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Model Evaluation — Cancer Trial Outcome Predictor", fontsize=13, fontweight="bold")

# 1. Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=["Failure", "Success"]
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix")

# 2. ROC curve
RocCurveDisplay.from_estimator(full_pipeline, X_test, y_test, ax=axes[1])
axes[1].plot([0,1],[0,1],"k--", label="Random baseline")
axes[1].set_title(f"ROC Curve  (AUC={roc_auc:.3f})")
axes[1].legend(fontsize=9)

# 3. Feature importances (top 15)
ohe_names = (full_pipeline.named_steps["preprocessor"]
             .named_transformers_["cat"]
             .get_feature_names_out(CATEGORICAL_COLS))
all_names  = list(ohe_names) + NUMERICAL_COLS
imps       = full_pipeline.named_steps["classifier"].feature_importances_
fi_df = (pd.DataFrame({"feature": all_names, "importance": imps})
           .sort_values("importance", ascending=True)
           .tail(15))
axes[2].barh(fi_df["feature"], fi_df["importance"], color="steelblue")
axes[2].set_title("Top 15 Feature Importances")
axes[2].set_xlabel("Importance score")

plt.tight_layout()
plt.savefig("/content/model_evaluation.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅  Saved: /content/model_evaluation.png")


## 💾 Cell 11 — Save model artifacts


In [ ]:
# ══════════════════════════════════════════════════════════════════
# ARTIFACT 1:  model_pipeline.pkl
#   Contains the FULL sklearn Pipeline:
#     preprocessor (OneHotEncoder + StandardScaler) + GBM classifier
#   This is the ONLY model file the Flask app needs.
#   There is NO separate scaler.pkl — the scaler lives inside this file.
# ══════════════════════════════════════════════════════════════════
pipeline_path = "/content/model_pipeline.pkl"
joblib.dump(full_pipeline, pipeline_path)
print(f"✅  Saved: {pipeline_path}")

# ══════════════════════════════════════════════════════════════════
# ARTIFACT 2:  feature_columns.json
#   Stores the ordered feature list and valid category values.
#   The Flask app reads this at startup to build the HTML form
#   and to guarantee the same column order at prediction time.
# ══════════════════════════════════════════════════════════════════
meta = {
    "feature_columns" : ALL_FEATURE_COLS,
    "categorical_cols": CATEGORICAL_COLS,
    "numerical_cols"  : NUMERICAL_COLS,
    "cancer_types"    : CANCER_TYPES,
    "drug_types"      : DRUG_TYPES,
    "phases"          : PHASES,
    "sponsor_types"   : SPONSOR_TYPES,
}
meta_path = "/content/feature_columns.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"✅  Saved: {meta_path}")

# ══════════════════════════════════════════════════════════════════
# ARTIFACT 3:  training_dataset.csv
#   The exact dataset used to train the model.
# ══════════════════════════════════════════════════════════════════
csv_path = "/content/training_dataset_synthetic.csv"
df_synth.to_csv(csv_path, index=False)
print(f"✅  Saved: {csv_path}")

# ══════════════════════════════════════════════════════════════════
# HOW TO EXTRACT scaler.pkl SEPARATELY (if your Flask app needs it)
# ══════════════════════════════════════════════════════════════════
scaler = full_pipeline.named_steps["preprocessor"].named_transformers_["num"]
scaler_path = "/content/scaler.pkl"
joblib.dump(scaler, scaler_path)
print(f"✅  Saved: {scaler_path}  (extracted from pipeline — same object)")

# Print sizes
for p in [pipeline_path, meta_path, csv_path, scaler_path]:
    kb = os.path.getsize(p) / 1024
    print(f"   {os.path.basename(p):35s}  {kb:.1f} KB")


## ⬇️ Cell 12 — Download files from Colab


In [ ]:
try:
    from google.colab import files
    for path in [
        "/content/model_pipeline.pkl",
        "/content/feature_columns.json",
        "/content/scaler.pkl",
        "/content/training_dataset_synthetic.csv",
        "/content/model_evaluation.png",
    ]:
        files.download(path)
        print(f"  Downloaded: {os.path.basename(path)}")
    print("\n✅  All files downloaded.")
except ImportError:
    print("Not running in Colab — files are at /content/")


## ✅ Cell 13 — Reload artifacts & run a prediction


In [ ]:
# Reload from disk to prove the saved file works correctly
pipe_loaded = joblib.load("/content/model_pipeline.pkl")
with open("/content/feature_columns.json") as f:
    meta_loaded = json.load(f)

# Single prediction example
example = pd.DataFrame([{
    "cancer_type"           : "breast",
    "drug_type"             : "targeted_therapy",
    "phase"                 : "III",
    "sponsor_type"          : "industry",
    "enrollment_size"       : 800.0,
    "trial_duration_months" : 60.0,
}])[meta_loaded["feature_columns"]]

prob  = pipe_loaded.predict_proba(example)[0, 1]
label = "LIKELY TO SUCCEED" if prob >= 0.5 else "HIGH RISK OF FAILURE"
risk  = "Low Risk" if prob >= 0.7 else ("Medium Risk" if prob >= 0.4 else "High Risk")

print("── Reload verification ──────────────────────────────")
print(f"  Model loaded from disk : ✅")
print(f"  Input  : Breast · Targeted therapy · Phase III · 800 pts · 60 mo · Industry")
print(f"  P(success)  = {prob:.4f}  ({prob*100:.1f}%)")
print(f"  Prediction  : {label}")
print(f"  Risk level  : {risk}")
print()
print("── Summary ──────────────────────────────────────────")
print(f"  Training rows    : {len(X_train):,}")
print(f"  Test rows        : {len(X_test):,}")
print(f"  Accuracy         : {acc:.4f}")
print(f"  Precision        : {prec:.4f}")
print(f"  Recall           : {rec:.4f}")
print(f"  F1 Score         : {f1:.4f}")
print(f"  ROC-AUC          : {roc_auc:.4f}")
print(f"  CV ROC-AUC mean  : {cv_aucs.mean():.4f} ± {cv_aucs.std():.4f}")
print()
print("  Drop model_pipeline.pkl + feature_columns.json into")
print("  the Flask app's  model/  folder to deploy.")


---
# PART B — Upgrade to REAL ClinicalTrials.gov data

Run this section **after Part A** if you want to train on real-world trial records.  
The output artifacts are drop-in replacements for the Flask app.  
No API key needed — ClinicalTrials.gov is a public database.

**Label used:** `hasResults` — whether the completed trial submitted results to the registry.  
This is the standard proxy label in published ML papers on trial outcome prediction  
(Lo et al. 2019; Fogel et al. 2018).


## 🌐 Cell 14 — Fetch real data from ClinicalTrials.gov API v2


In [ ]:
# ClinicalTrials.gov API v2
# Public REST API — no authentication required
# Rate limit: ~50 requests/minute  →  we sleep 1s between pages
# Max page size: 1000 records per request

BASE_URL   = "https://clinicaltrials.gov/api/v2/studies"
PAGE_SIZE  = 1000
MAX_PAGES  = 6   # increase for more data (6 pages × ~8 conditions ≈ 48,000 records max)

CANCER_QUERIES = [
    "lung cancer", "breast cancer", "colorectal cancer",
    "prostate cancer", "leukemia", "melanoma",
    "pancreatic cancer", "ovarian cancer",
]

def fetch_trials(condition, max_pages=MAX_PAGES):
    records, page_token, page_num = [], None, 0
    while page_num < max_pages:
        params = {
            "query.cond"          : condition,
            "filter.studyType"    : "INTERVENTIONAL",
            "filter.overallStatus": "COMPLETED",
            "pageSize"            : PAGE_SIZE,
            "format"              : "json",
        }
        if page_token:
            params["pageToken"] = page_token
        try:
            resp = requests.get(BASE_URL, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except Exception as exc:
            print(f"  ⚠ Failed for '{condition}' page {page_num}: {exc}")
            break
        studies    = data.get("studies", [])
        records.extend(studies)
        page_token = data.get("nextPageToken")
        page_num  += 1
        print(f"  '{condition}'  page {page_num}  fetched {len(studies)}  total {len(records)}")
        if not page_token:
            break
        time.sleep(1.0)
    return records

print("Fetching COMPLETED cancer trials from ClinicalTrials.gov …")
print("(~2-3 minutes for all 8 cancer types)")
all_raw = []
for cond in CANCER_QUERIES:
    all_raw.extend(fetch_trials(cond))
    time.sleep(2)

print(f"\n✅  Total raw records: {len(all_raw):,}")


## 🔍 Cell 15 — Parse, clean & engineer features from real data


In [ ]:
from datetime import datetime

def safe(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict): return default
        d = d.get(k, default)
        if d is None: return default
    return d

def parse_phase(phases):
    m = {"PHASE1":"I","PHASE2":"II","PHASE3":"III","PHASE4":"IV","EARLY_PHASE1":"I"}
    return m.get((phases or [""])[0], None)

def parse_sponsor(cls):
    m = {"INDUSTRY":"industry","NIH":"government","FED":"government"}
    return m.get(str(cls).upper(), "academic")

def parse_drug(interventions):
    if not interventions: return None
    itype = str(safe(interventions[0],"type") or "").upper()
    name  = str(safe(interventions[0],"name") or "").lower()
    base  = {"DRUG":"chemotherapy","BIOLOGICAL":"immunotherapy",
             "RADIATION":"radiation","COMBINATION_PRODUCT":"combination"}.get(itype)
    if base == "chemotherapy":
        if any(k in name for k in ["mab","nib","tinib","targeted","inhibitor","kinase","trastuzumab","bevacizumab"]):
            base = "targeted_therapy"
        elif any(k in name for k in ["tamoxifen","letrozole","anastrozole","hormone","fulvestrant","bicalutamide"]):
            base = "hormone_therapy"
        elif any(k in name for k in ["pembrolizumab","nivolumab","ipilimumab","checkpoint","pd-1","pd-l1"]):
            base = "immunotherapy"
    return base

def parse_cancer(conditions):
    text = " ".join(conditions or []).lower()
    for label, kws in [
        ("lung",["lung","nsclc","sclc"]),("breast",["breast"]),
        ("colorectal",["colorectal","colon","rectal"]),("prostate",["prostate"]),
        ("leukemia",["leukemia","aml","cll","cml"]),("melanoma",["melanoma"]),
        ("pancreatic",["pancreatic","pancreas"]),("ovarian",["ovarian","ovary"]),
        ("bladder",["bladder","urothelial"]),("renal",["renal","kidney"]),
        ("lymphoma",["lymphoma"]),("liver",["hepatocellular","liver cancer"]),
    ]:
        if any(k in text for k in kws): return label
    return None

def months_between(s, e):
    for fmt in ["%Y-%m-%d","%Y-%m","%Y"]:
        try:
            sd = datetime.strptime(str(s)[:10], fmt)
            ed = datetime.strptime(str(e)[:10], fmt)
            if ed > sd: return round((ed-sd).days/30.44, 1)
        except: pass
    return None

def parse_study(study):
    ps  = study.get("protocolSection", {})
    sta = ps.get("statusModule", {})
    des = ps.get("designModule", {})
    spo = ps.get("sponsorCollaboratorsModule", {})
    con = ps.get("conditionsModule", {})
    arm = ps.get("armsInterventionsModule", {})
    if sta.get("overallStatus") != "COMPLETED": return None

    phase        = parse_phase(des.get("phases", []))
    enroll       = safe(des, "enrollmentInfo", "count")
    try: enroll  = float(enroll)
    except: enroll = None
    start        = safe(sta, "startDateStruct", "date")
    end          = safe(sta, "completionDateStruct", "date")
    duration     = months_between(start, end)
    sponsor_type = parse_sponsor(safe(spo, "leadSponsor", "class"))
    cancer_type  = parse_cancer(con.get("conditions", []))
    drug_type    = parse_drug(arm.get("interventions", []))
    outcome      = 1 if study.get("hasResults", False) else 0

    if not all([phase, enroll, duration, cancer_type, drug_type]):
        return None
    return dict(cancer_type=cancer_type, drug_type=drug_type, phase=phase,
                enrollment_size=float(enroll),
                trial_duration_months=float(duration),
                sponsor_type=sponsor_type, outcome=outcome)

rows_real = [r for s in all_raw if (r := parse_study(s))]
df_real   = pd.DataFrame(rows_real).drop_duplicates()

# Clean
df_real = df_real[df_real["phase"].isin(["I","II","III","IV"])]
df_real["enrollment_size"]       = df_real["enrollment_size"].clip(5, 10000)
df_real["trial_duration_months"] = df_real["trial_duration_months"].clip(1, 240)

print(f"✅  Real dataset: {len(df_real):,} rows")
print(f"\nOutcome balance:\n{df_real['outcome'].value_counts()}")
print(f"\nCancer types:\n{df_real['cancer_type'].value_counts()}")
print(f"\nPhases:\n{df_real['phase'].value_counts()}")


## 🤖 Cell 16 — Train & evaluate model on real data


In [ ]:
X_r = df_real[ALL_FEATURE_COLS]
y_r = df_real["outcome"]

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_r, y_r, test_size=0.2, random_state=42, stratify=y_r
)

real_pipeline = Pipeline(steps=[
    ("preprocessor", ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ("num", StandardScaler(), NUMERICAL_COLS),
    ], remainder="drop")),
    ("classifier", GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.05,
        max_depth=4, subsample=0.8, random_state=42,
    )),
])

real_pipeline.fit(X_tr_r, y_tr_r)
yp    = real_pipeline.predict(X_te_r)
ypp   = real_pipeline.predict_proba(X_te_r)[:,1]

r_acc = accuracy_score (y_te_r, yp)
r_prec= precision_score(y_te_r, yp,  zero_division=0)
r_rec = recall_score   (y_te_r, yp,  zero_division=0)
r_f1  = f1_score       (y_te_r, yp,  zero_division=0)
r_auc = roc_auc_score  (y_te_r, ypp)

print("╔══════════════════════════════════════════╗")
print("║  REAL DATA MODEL — TEST SET RESULTS      ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Accuracy   : {r_acc:.4f}  ({r_acc*100:.2f}%)            ║")
print(f"║  Precision  : {r_prec:.4f}                        ║")
print(f"║  Recall     : {r_rec:.4f}                        ║")
print(f"║  F1 Score   : {r_f1:.4f}                        ║")
print(f"║  ROC-AUC    : {r_auc:.4f}                        ║")
print("╚══════════════════════════════════════════╝")
print()
print(classification_report(y_te_r, yp,
      target_names=["Failure (0)","Success (1)"], zero_division=0))

# Save real-data artifacts
joblib.dump(real_pipeline, "/content/model_pipeline_real.pkl")
real_meta = {
    "feature_columns" : ALL_FEATURE_COLS,
    "categorical_cols": CATEGORICAL_COLS,
    "numerical_cols"  : NUMERICAL_COLS,
    "cancer_types"    : sorted(df_real["cancer_type"].dropna().unique().tolist()),
    "drug_types"      : sorted(df_real["drug_type"].dropna().unique().tolist()),
    "phases"          : ["I","II","III","IV"],
    "sponsor_types"   : sorted(df_real["sponsor_type"].dropna().unique().tolist()),
    "data_source"     : "ClinicalTrials.gov API v2",
    "label_note"      : "hasResults=True → Success(1), False → Failure(0)",
    "n_train"         : int(len(X_tr_r)),
    "metrics"         : {"accuracy":round(r_acc,4),"precision":round(r_prec,4),
                         "recall":round(r_rec,4),"f1":round(r_f1,4),"roc_auc":round(r_auc,4)},
}
with open("/content/feature_columns_real.json","w") as f:
    json.dump(real_meta, f, indent=2)
df_real.to_csv("/content/training_dataset_real.csv", index=False)

print("✅  Real-data artifacts saved.")
print("    Rename model_pipeline_real.pkl  → model_pipeline.pkl")
print("    Rename feature_columns_real.json → feature_columns.json")
print("    Drop both into the Flask app's  model/  folder.")

try:
    from google.colab import files
    for p in ["/content/model_pipeline_real.pkl",
              "/content/feature_columns_real.json",
              "/content/training_dataset_real.csv"]:
        files.download(p)
except ImportError:
    pass
